# Berkeley Circle 24h Current Forecasts — SF Bay Map Visualization

Loads the best ensemble model (XGBoost + LSTM) from MLflow, generates predictions
over the 2025 hold-out test set, and overlays predictions and errors on a map
of SF Bay zoomed to the Berkeley Circle HFR measurement point.

---
**Four figures:**
1. **Context map** — where Berkeley Circle sits in SF Bay
2. **Quiver fan** — hourly current vectors (observed vs predicted) over a 2-week window
3. **Monthly error summary** — mean predicted current + MAE for each month of 2025
4. **Current rose comparison** — directional distribution of observed vs predicted currents

## 1  Setup

In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import mlflow
import mlflow.xgboost

warnings.filterwarnings('ignore')

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))

from src.config import (
    HFR_LAT, HFR_LON, TRAIN_END, TEST_START, FORECAST_HOURS, TARGET_COLS,
    MLFLOW_EXPERIMENT, BEGIN_ISO, END_ISO, TRAIN_CSV_PATH,
)
from src.etl.features import get_feature_cols

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
})
print('REPO:', REPO)
print('HFR point:', HFR_LAT, HFR_LON)

## 2  Load Models & Artifacts

In [ ]:
mlflow.set_tracking_uri(str(REPO / 'mlruns'))

# Grab most recent completed run
runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string="tags.status = 'completed'",
    order_by=['start_time DESC'],
    max_results=1,
)
assert not runs.empty, 'No completed runs found — run python src/train.py first'
run_id        = runs.iloc[0]['run_id']
run_name      = runs.iloc[0]['tags.mlflow.runName']
experiment_id = runs.iloc[0]['experiment_id']
print(f'Best run: {run_name}  ({run_id[:8]}...)')

# Locate inference artifacts: try run-specific path first, then stable data/artifacts/
artifact_dir = REPO / f'mlruns/{experiment_id}/{run_id}/artifacts/inference_artifacts'
if not artifact_dir.exists():
    artifact_dir = REPO / 'data/artifacts'
print('Artifact dir:', artifact_dir)

with open(artifact_dir / 'pca_angle.pkl', 'rb') as f:
    pca_angle = pickle.load(f)
with open(artifact_dir / 'feat_scaler.pkl', 'rb') as f:
    feat_scaler = pickle.load(f)
with open(artifact_dir / 'target_scaler.pkl', 'rb') as f:
    target_scaler = pickle.load(f)

print(f'PCA rotation angle: {np.degrees(pca_angle):.1f}°  (from east)')

In [ ]:
# Discover model IDs for this run (name-based lookup via LoggedModel API)
client = mlflow.tracking.MlflowClient()
experiment_id = runs.iloc[0]['experiment_id']

model_uris = {}
try:
    logged_models = client.search_logged_models(experiment_ids=[experiment_id])
    for m in logged_models:
        if m._source_run_id == run_id:
            model_uris[m._name] = m._model_uri
except Exception:
    model_uris = {
        'xgb_along': f'runs:/{run_id}/xgb_along',
        'xgb_cross':  f'runs:/{run_id}/xgb_cross',
    }

print('Found models:', list(model_uris.keys()))

xgb_along = mlflow.xgboost.load_model(model_uris['xgb_along'])
xgb_cross  = mlflow.xgboost.load_model(model_uris['xgb_cross'])
print('XGBoost models loaded')

## 3  Load Dataset & Generate Predictions

In [ ]:
csv_path = REPO / TRAIN_CSV_PATH.format(
    begin=BEGIN_ISO.replace('-', ''),
    end=END_ISO.replace('-', ''),
)
df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
feature_cols = get_feature_cols(df)
print(f'Dataset: {len(df):,} rows × {df.shape[1]} cols | {len(feature_cols)} features')

train = df.loc[:TRAIN_END]
test  = df.loc[TEST_START:].dropna(subset=TARGET_COLS)
print(f'Train: {len(train):,} rows ({train.index[0].date()} – {train.index[-1].date()})')
print(f'Test:  {len(test):,} rows  ({test.index[0].date()} – {test.index[-1].date()})')

In [ ]:
fh = FORECAST_HOURS
X_te = test[feature_cols]

pred_along = pd.Series(xgb_along.predict(X_te), index=test.index)
pred_cross  = pd.Series(xgb_cross.predict(X_te),  index=test.index)
model_label = 'XGBoost'
print(f'Predictions done — {len(pred_along):,} timesteps')

In [ ]:
# Reconstruct full current = model residual + tidal component
tide_a = test[f'tide_curr_along_{fh}h']
tide_c = test[f'tide_curr_cross_{fh}h']
true_a = test['resid_along_fwd']
true_c = test['resid_cross_fwd']

pred_full_a = pred_along + tide_a
pred_full_c = pred_cross  + tide_c
obs_full_a  = true_a + tide_a
obs_full_c  = true_c + tide_c

# Rotate from along/cross back to geographic u (east) / v (north)
def along_cross_to_uv(along, cross, angle):
    c, s = np.cos(angle), np.sin(angle)
    return c * along - s * cross, s * along + c * cross

pred_u, pred_v = along_cross_to_uv(pred_full_a, pred_full_c, pca_angle)
obs_u,  obs_v  = along_cross_to_uv(obs_full_a,  obs_full_c,  pca_angle)

pred_spd    = np.sqrt(pred_u**2 + pred_v**2)
obs_spd     = np.sqrt(obs_u**2  + obs_v**2)
speed_err   = pred_spd - obs_spd
abs_spd_err = speed_err.abs()

results = pd.DataFrame({
    'obs_u': obs_u,   'obs_v': obs_v,   'obs_spd': obs_spd,
    'pred_u': pred_u, 'pred_v': pred_v, 'pred_spd': pred_spd,
    'speed_err': speed_err, 'abs_spd_err': abs_spd_err,
    'err_u': pred_u - obs_u, 'err_v': pred_v - obs_v,
})

overall_mae   = abs_spd_err.mean()
overall_skill = 1 - abs_spd_err.mean() / obs_spd.mean()
print(f'Speed MAE: {overall_mae:.3f} m/s  |  Skill: {overall_skill:.3f}')
results.head()

## 4  Map Visualizations

All figures use Cartopy with Natural Earth features.
The Berkeley Circle HFR pixel is at **37.856 °N, 122.364 °W** — a single
measurement point in the main channel between Richmond and the East Bay.

In [ ]:
# ── Shared cartopy helpers ────────────────────────────────────────────────────

PROJ = ccrs.PlateCarree()

# Known station locations
STATIONS = {
    'Richmond\n(tide gauge)':  (37.910, -122.365),
    'Fort Point\n(tide gauge)': (37.807, -122.478),
}

def _base_map(ax, extent, land_color='#e8dcc8', ocean_color='#b8d4e8',
              resolution='10m', gridlines=True):
    """Add land/ocean/coastline features to a cartopy axes."""
    ax.set_extent(extent, crs=PROJ)
    ax.add_feature(cfeature.LAND.with_scale(resolution),
                   facecolor=land_color, zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale(resolution),
                   facecolor=ocean_color, zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale(resolution),
                   linewidth=0.6, edgecolor='#555555', zorder=1)
    ax.add_feature(cfeature.RIVERS.with_scale(resolution),
                   linewidth=0.4, edgecolor='#7ab', zorder=1)
    if gridlines:
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                          alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
    return ax


def _add_scale_arrow(ax, lon, lat, u_ms, label, color='k', fontsize=8):
    """Draw a labeled reference arrow for current speed scale."""
    scale = 0.04  # degrees per m/s
    ax.annotate('', xy=(lon + u_ms * scale, lat),
                xytext=(lon, lat), xycoords=PROJ._as_mpl_transform(ax),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5),
                annotation_clip=False)
    ax.text(lon + u_ms * scale / 2, lat - 0.005, label,
            ha='center', va='top', fontsize=fontsize, color=color,
            transform=PROJ)

print('Helpers defined.')

### Figure 1 — SF Bay Context Map

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8),
                       subplot_kw={'projection': PROJ})

_base_map(ax, extent=[-123.1, -121.9, 37.5, 38.3])

# HFR measurement point (main)
ax.plot(HFR_LON, HFR_LAT, marker='*', markersize=18, color='#e74c3c',
        transform=PROJ, zorder=5, label='Berkeley Circle HFR pixel')
ax.annotate('Berkeley Circle\n(HFR pixel, this study)',
            xy=(HFR_LON, HFR_LAT), xytext=(HFR_LON + 0.15, HFR_LAT + 0.05),
            transform=PROJ, fontsize=9, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

# Tide gauge stations
for name, (lat, lon) in STATIONS.items():
    ax.plot(lon, lat, marker='D', markersize=8, color='steelblue',
            transform=PROJ, zorder=5)
    ax.annotate(name, xy=(lon, lat), xytext=(lon + 0.06, lat - 0.03),
                transform=PROJ, fontsize=7.5, color='steelblue',
                arrowprops=dict(arrowstyle='-', color='steelblue', lw=0.8))

# Point Reyes (off-map, label only)
ax.text(-122.85, 38.22, 'Point Reyes\n(tide gauge, off map →)',
        fontsize=7.5, color='steelblue', transform=PROJ, ha='right')

# Bay labels
label_kw = dict(transform=PROJ, ha='center', va='center',
                style='italic', color='#1a5276')
ax.text(-122.42, 37.75, 'San Francisco Bay', fontsize=9, **label_kw)
ax.text(-122.45, 38.06, 'San Pablo Bay',     fontsize=9, **label_kw)
ax.text(-122.22, 38.08, 'Suisun Bay',        fontsize=9, **label_kw)
ax.text(-122.1,  37.95, 'Carquinez\nStrait', fontsize=8, **label_kw)
ax.text(-122.60, 37.72, 'Golden Gate', transform=PROJ, ha='center',
        fontsize=8.5, color='#555', style='italic')

# City labels
city_kw = dict(transform=PROJ, fontsize=8, color='#333')
ax.text(-122.27, 37.87, '● Berkeley',      **city_kw)
ax.text(-122.17, 37.85, '● Oakland',       **city_kw)
ax.text(-122.35, 37.95, '● Richmond',      **city_kw)
ax.text(-122.42, 37.79, '● San Francisco', **city_kw)

# Legend
legend_handles = [
    mpatches.Patch(color='#e74c3c', label='Berkeley Circle HFR pixel (37.856°N, 122.364°W)'),
    mpatches.Patch(color='steelblue', label='NOAA tide gauge stations'),
]
ax.legend(handles=legend_handles, loc='lower left', fontsize=8.5, framealpha=0.9)

ax.set_title('SF Bay — Berkeley Circle Measurement Location & Predictor Stations',
             fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

### Figure 2 — Prediction Quiver Fan (Feb 2025)

Each arrow originates from the Berkeley Circle HFR pixel and points in the direction
of the hourly current (sampled every 6 h). **Blue** = observed; **orange** = ensemble
predicted. Arrow length is proportional to current speed.

In [ ]:
# ── Select Feb 1–14, 2025 sampled every 6 h ──────────────────────────────────
mask = (results.index >= '2025-02-01') & (results.index < '2025-02-15')
sample = results[mask].iloc[::6].dropna(subset=['obs_u', 'pred_u'])
n = len(sample)
print(f'Plotting {n} arrow pairs  ({sample.index[0]} – {sample.index[-1]})')

# Scale: 1 m/s current → 0.04° on map
SCALE = 0.04

fig, ax = plt.subplots(figsize=(9, 8), subplot_kw={'projection': PROJ})
_base_map(ax, extent=[-122.55, -122.18, 37.78, 37.97])

# Color observations by time (sequential blue)
t_norm = (sample.index - sample.index[0]).total_seconds()
t_norm = (t_norm - t_norm.min()) / (t_norm.max() - t_norm.min() + 1e-9)
obs_colors  = cm.Blues(0.35 + 0.6 * t_norm)

# Color predictions by absolute speed error (green → yellow → red)
err_norm = Normalize(vmin=0, vmax=0.25)
pred_colors = cm.RdYlGn_r(err_norm(sample['abs_spd_err'].clip(0, 0.25)))

# Draw arrows from Richmond point
for i, (t, row) in enumerate(sample.iterrows()):
    # Observed arrow
    du_o, dv_o = row['obs_u'] * SCALE, row['obs_v'] * SCALE
    ax.annotate('',
                xy=(HFR_LON + du_o, HFR_LAT + dv_o),
                xytext=(HFR_LON, HFR_LAT),
                xycoords=PROJ._as_mpl_transform(ax),
                textcoords=PROJ._as_mpl_transform(ax),
                arrowprops=dict(arrowstyle='->', color=obs_colors[i],
                                lw=1.6, mutation_scale=12),
                annotation_clip=False, zorder=3)

    # Predicted arrow (slightly thinner)
    du_p, dv_p = row['pred_u'] * SCALE, row['pred_v'] * SCALE
    ax.annotate('',
                xy=(HFR_LON + du_p, HFR_LAT + dv_p),
                xytext=(HFR_LON, HFR_LAT),
                xycoords=PROJ._as_mpl_transform(ax),
                textcoords=PROJ._as_mpl_transform(ax),
                arrowprops=dict(arrowstyle='->', color=pred_colors[i],
                                lw=1.1, mutation_scale=9, alpha=0.85),
                annotation_clip=False, zorder=4)

# Richmond point marker
ax.plot(HFR_LON, HFR_LAT, 'ko', markersize=6, transform=PROJ, zorder=6)
ax.text(HFR_LON + 0.005, HFR_LAT - 0.009, 'Richmond\nReach',
        transform=PROJ, fontsize=8, ha='left', color='k',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

# Colorbars
sm_time = cm.ScalarMappable(cmap='Blues',
                             norm=Normalize(0, (sample.index[-1] - sample.index[0]).days))
sm_time.set_array([])
cb1 = fig.colorbar(sm_time, ax=ax, orientation='vertical', fraction=0.03,
                    pad=0.02, shrink=0.55)
cb1.set_label('Observed — days into period', fontsize=8)
cb1.ax.tick_params(labelsize=7)

sm_err = cm.ScalarMappable(cmap='RdYlGn_r', norm=err_norm)
sm_err.set_array([])
cb2 = fig.colorbar(sm_err, ax=ax, orientation='vertical', fraction=0.03,
                    pad=0.09, shrink=0.55)
cb2.set_label('Predicted — speed error (m/s)', fontsize=8)
cb2.ax.tick_params(labelsize=7)

# Scale reference arrow
ref_lon, ref_lat = -122.53, 37.80
ref_spd = 0.4  # m/s
ax.annotate('',
            xy=(ref_lon + ref_spd * SCALE, ref_lat),
            xytext=(ref_lon, ref_lat),
            xycoords=PROJ._as_mpl_transform(ax),
            textcoords=PROJ._as_mpl_transform(ax),
            arrowprops=dict(arrowstyle='->', color='k', lw=2),
            annotation_clip=False, zorder=6)
ax.text(ref_lon + ref_spd * SCALE / 2, ref_lat - 0.007,
        f'{ref_spd:.1f} m/s', ha='center', va='top', fontsize=8,
        transform=PROJ, fontweight='bold')

# Legend
leg_handles = [
    mpatches.Patch(color='steelblue', label='Observed (blue = earlier → later)'),
    mpatches.Patch(color='#66bb6a',   label='Predicted — low error (green)'),
    mpatches.Patch(color='#ef5350',   label='Predicted — high error (red)'),
]
ax.legend(handles=leg_handles, loc='upper right', fontsize=8, framealpha=0.9)

ax.set_title(
    f'24h-ahead Current Forecasts — Berkeley Circle  |  Feb 1–14 2025  |  {model_label}\n'
    'Each arrow = one 6-h sample, origin = HFR pixel, length ∝ speed',
    fontsize=10, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()

### Figure 3 — Monthly Predicted Current & Error (2025 Test Set)

Each panel shows the monthly **mean predicted current vector** (arrow, length ∝ speed)
at the Berkeley Circle point.  Arrow colour encodes the monthly speed **MAE**;
the grey background shows the inter-quartile spread of predicted speed.

In [ ]:
months_in_test = sorted(results.index.to_period('M').unique())
n_months = len(months_in_test)
print(f'{n_months} months in test set: {months_in_test[0]} – {months_in_test[-1]}')

# Compute monthly stats
monthly = []
for period in months_in_test:
    sub = results[results.index.to_period('M') == period]
    if len(sub) < 24:
        continue
    monthly.append({
        'period': period,
        'label': period.strftime('%b %Y'),
        'mean_pred_u': sub['pred_u'].mean(),
        'mean_pred_v': sub['pred_v'].mean(),
        'mean_obs_u':  sub['obs_u'].mean(),
        'mean_obs_v':  sub['obs_v'].mean(),
        'mae': sub['abs_spd_err'].mean(),
        'pred_spd_q25': sub['pred_spd'].quantile(0.25),
        'pred_spd_q75': sub['pred_spd'].quantile(0.75),
    })
monthly = pd.DataFrame(monthly)
print(monthly[['label','mae','mean_pred_u','mean_pred_v']].round(3).to_string(index=False))

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(monthly) / n_cols))

EXTENT = [-122.55, -122.18, 37.78, 37.97]  # Berkeley Circle zoom
SCALE  = 0.04  # degrees per m/s

mae_norm  = Normalize(vmin=monthly['mae'].min() * 0.9, vmax=monthly['mae'].max() * 1.05)
mae_cmap  = cm.RdYlGn_r

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4 * n_cols, 3.3 * n_rows),
    subplot_kw={'projection': PROJ},
)
axes_flat = axes.flatten()

for idx, row in monthly.iterrows():
    ax = axes_flat[idx]
    _base_map(ax, extent=EXTENT, gridlines=False)

    mae_color = mae_cmap(mae_norm(row['mae']))

    # IQR band as a faint circle (speed spread)
    r_inner = row['pred_spd_q25'] * SCALE
    r_outer = row['pred_spd_q75'] * SCALE
    circle_inner = plt.Circle((HFR_LON, HFR_LAT), r_inner,
                               color='lightgrey', fill=True, alpha=0.5,
                               transform=PROJ, zorder=2)
    circle_outer = plt.Circle((HFR_LON, HFR_LAT), r_outer,
                               color='lightgrey', fill=True, alpha=0.25,
                               transform=PROJ, zorder=2)
    ax.add_patch(circle_outer)
    ax.add_patch(circle_inner)

    # Mean observed arrow (grey)
    ax.annotate('',
                xy=(HFR_LON + row['mean_obs_u'] * SCALE,
                    HFR_LAT + row['mean_obs_v'] * SCALE),
                xytext=(HFR_LON, HFR_LAT),
                xycoords=PROJ._as_mpl_transform(ax),
                textcoords=PROJ._as_mpl_transform(ax),
                arrowprops=dict(arrowstyle='->', color='#666666',
                                lw=2.0, mutation_scale=14),
                annotation_clip=False, zorder=4)

    # Mean predicted arrow (coloured by MAE)
    ax.annotate('',
                xy=(HFR_LON + row['mean_pred_u'] * SCALE,
                    HFR_LAT + row['mean_pred_v'] * SCALE),
                xytext=(HFR_LON, HFR_LAT),
                xycoords=PROJ._as_mpl_transform(ax),
                textcoords=PROJ._as_mpl_transform(ax),
                arrowprops=dict(arrowstyle='->', color=mae_color,
                                lw=2.5, mutation_scale=16),
                annotation_clip=False, zorder=5)

    ax.plot(HFR_LON, HFR_LAT, 'ko', markersize=5, transform=PROJ, zorder=6)

    ax.set_title(f"{row['label']}  |  MAE={row['mae']:.3f} m/s",
                 fontsize=9, fontweight='bold', pad=3)

# Hide unused panels
for ax in axes_flat[len(monthly):]:
    ax.set_visible(False)

# Shared colorbar
sm = cm.ScalarMappable(cmap=mae_cmap, norm=mae_norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=axes_flat[:len(monthly)], orientation='vertical',
                   shrink=0.6, pad=0.02, fraction=0.015)
cb.set_label('Speed MAE (m/s)', fontsize=9)

fig.suptitle(
    f'Monthly Mean Current at Berkeley Circle — 2025 Test Set ({model_label})\n'
    'Coloured arrow = predicted (colour = MAE)  |  Grey arrow = observed  |  '
    'Shaded rings = IQR of predicted speed',
    fontsize=11, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

### Figure 4 — Current Rose: Observed vs Predicted

Directional frequency of current speed, overlaid on the Berkeley Circle context map.
The roses show whether the model captures the dominant ebb/flood axes correctly.

In [ ]:
def _current_rose(u, v, ax_polar, title, color, alpha=0.75):
    """Stacked bar current rose on a polar axes."""
    spd  = np.sqrt(u**2 + v**2)
    dirn = (np.degrees(np.arctan2(u, v)) + 360) % 360  # met convention: clockwise from N
    # Bin into 16 direction sectors × 4 speed bands
    dir_bins = np.linspace(0, 360, 17)
    spd_bins = [0, 0.1, 0.25, 0.45, np.inf]
    spd_labels = ['< 0.1', '0.1–0.25', '0.25–0.45', '> 0.45']
    bar_colors = [plt.cm.Blues(x) for x in [0.3, 0.5, 0.7, 0.9]]

    n_dir = len(dir_bins) - 1
    width = 2 * np.pi / n_dir
    theta = np.deg2rad(dir_bins[:-1] + 360 / n_dir / 2)

    bottom = np.zeros(n_dir)
    for k, (s_lo, s_hi) in enumerate(zip(spd_bins[:-1], spd_bins[1:])):
        mask_spd = (spd >= s_lo) & (spd < s_hi)
        counts = np.array([
            np.sum(mask_spd & (dirn >= dir_bins[j]) & (dirn < dir_bins[j+1]))
            for j in range(n_dir)
        ], dtype=float)
        pct = 100 * counts / len(spd)
        ax_polar.bar(theta, pct, width=width * 0.9, bottom=bottom,
                     color=bar_colors[k], alpha=alpha,
                     label=spd_labels[k])
        bottom += pct

    ax_polar.set_theta_zero_location('N')
    ax_polar.set_theta_direction(-1)
    ax_polar.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax_polar.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
    ax_polar.set_title(title, fontsize=10, fontweight='bold', pad=12)
    ax_polar.yaxis.set_tick_params(labelsize=7)
    ax_polar.set_rlabel_position(135)
    max_r = bottom.max()
    ax_polar.set_ylim(0, max_r * 1.1)
    ax_polar.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))
    return ax_polar


fig = plt.figure(figsize=(14, 6))

# Left: map context
ax_map = fig.add_subplot(1, 3, 1, projection=PROJ)
_base_map(ax_map, extent=[-122.7, -122.1, 37.7, 38.1], gridlines=True)
ax_map.plot(HFR_LON, HFR_LAT, marker='*', markersize=16, color='#e74c3c',
            transform=PROJ, zorder=5)
ax_map.set_title('Berkeley Circle location\nin SF Bay', fontsize=10, fontweight='bold')

# Middle: observed rose
ax_obs = fig.add_subplot(1, 3, 2, projection='polar')
_current_rose(results['obs_u'], results['obs_v'], ax_obs,
              'Observed Currents\n(HFR, 2025)', 'steelblue')

# Right: predicted rose
ax_pred = fig.add_subplot(1, 3, 3, projection='polar')
_current_rose(results['pred_u'], results['pred_v'], ax_pred,
              f'Predicted Currents\n({model_label}, 2025)', 'darkorange')

# Shared legend
handles = ax_obs.get_legend_handles_labels()[0]
labels  = ['< 0.1 m/s', '0.1–0.25 m/s', '0.25–0.45 m/s', '> 0.45 m/s']
fig.legend(handles, labels, title='Speed (m/s)', loc='lower center',
           ncol=4, fontsize=8.5, framealpha=0.9, bbox_to_anchor=(0.5, -0.06))

fig.suptitle(
    'Current Direction Rose — Berkeley Circle  |  2025 Test Set\n'
    'Radial axis = % of hours  |  Clockwise from North',
    fontsize=11, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

### Figure 5 — Speed Error Distribution by Tidal Phase

Binned scatter showing how prediction errors vary across the tidal cycle.
Overlaid on the Berkeley Circle map to contextualise where these currents occur.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: speed scatter (predicted vs observed)
ax1 = axes[0]
spd_max = max(results['obs_spd'].quantile(0.99), results['pred_spd'].quantile(0.99))
h = ax1.hist2d(results['obs_spd'], results['pred_spd'],
               bins=60, range=[[0, spd_max], [0, spd_max]],
               cmap='YlOrRd', norm=mcolors.LogNorm())
fig.colorbar(h[3], ax=ax1, label='Count (log scale)')
ax1.plot([0, spd_max], [0, spd_max], 'k--', lw=1.2, label='1:1 line')
ax1.set_xlabel('Observed speed (m/s)', fontsize=10)
ax1.set_ylabel('Predicted speed (m/s)', fontsize=10)
ax1.set_title(f'Speed: Predicted vs Observed\n'
              f'MAE = {overall_mae:.3f} m/s  |  Skill = {overall_skill:.3f}',
              fontsize=10, fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_aspect('equal')

# Right: signed speed error vs observed speed (bias analysis)
ax2 = axes[1]
sc = ax2.hexbin(results['obs_spd'], results['speed_err'],
                gridsize=50, cmap='RdBu_r', mincnt=1,
                extent=[0, spd_max, -0.5, 0.5])
fig.colorbar(sc, ax=ax2, label='Count')
ax2.axhline(0, color='k', lw=1.2, linestyle='--', label='No bias')
ax2.axhline(results['speed_err'].median(), color='darkorange', lw=1.5,
            linestyle='-', label=f'Median bias ({results["speed_err"].median():.3f} m/s)')
ax2.set_xlabel('Observed speed (m/s)', fontsize=10)
ax2.set_ylabel('Predicted − Observed speed (m/s)', fontsize=10)
ax2.set_title('Speed Bias vs Observed Speed\n'
              '(positive = over-prediction, negative = under-prediction)',
              fontsize=10, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim(-0.5, 0.5)

fig.suptitle(f'Prediction Quality — Berkeley Circle 2025 Test Set  ({model_label})',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5  Summary Statistics

In [ ]:
from src.evaluation.metrics import evaluate_model

tide_a_te = test[f'tide_curr_along_{fh}h'].reindex(results.index)
tide_c_te = test[f'tide_curr_cross_{fh}h'].reindex(results.index)
true_a_te = test['resid_along_fwd'].reindex(results.index).dropna()
true_c_te = test['resid_cross_fwd'].reindex(results.index).dropna()

metrics = evaluate_model(
    'xgboost',
    pred_along.reindex(true_a_te.index),
    pred_cross.reindex(true_c_te.index),
    true_a_te, true_c_te,
    tide_a_te, tide_c_te,
)

print('\n── 2025 Test Set Metrics ──')
for k, v in sorted(metrics.items()):
    print(f'  {k:<35} {v:.4f}')